# Solutions 02: Tokenizers, Alignment, and Real Model Outputs

Solutions to the five exercises of **Lab 02** (`lab-02-tokenizers-alignment-real-outputs`).
Four solutions are fully **live** (executed during the build, ending in asserts the
interpretation cells reference). Exercise 1 is **part live, part gated**: it asks for a
training-scale model pair, so the identical check is executed live on the lab's small pair
as the demonstration, and the training-scale code is written in full behind the
`RUN_TRAINING` flag below with expected results stated instead of fabricated. Attempt the
exercises yourself before opening this file.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

RUN_TRAINING = False    # set True on a machine that can hold Qwen3-8B; see Exercise 1

import sys, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

from kd_core import (kl_divergence, shift_for_next_token, completion_mask_from_prompt_lens,
                     topk_truncation_bias, bytes_per_token_cache, masked_mean)

torch.manual_seed(0)
DTYPE = torch.float32
TEACHER = "HuggingFaceTB/SmolLM2-360M-Instruct"
STUDENT = "HuggingFaceTB/SmolLM2-135M-Instruct"

tok = AutoTokenizer.from_pretrained(TEACHER)
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=DTYPE).eval()
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=DTYPE).eval()

# Rebuild the lab's section 2 batch exactly: correct padding, completion-only mask.
EOS_ID = tok.eos_token_id
PAD_ID = tok.convert_tokens_to_ids("<|endoftext|>")
pairs = [
    ("What is 17 * 23?", "17 * 23 = 391."),
    ("Name the largest planet in the solar system.", "Jupiter."),
    ("Write one sentence about autumn.", "The leaves turn amber and the air smells of rain."),
    ("In Python, how do I reverse a list in place?", "Call `my_list.reverse()`."),
]

def build_batch(pad_id):
    prompt_ids, full_ids = [], []
    for user_msg, completion in pairs:
        p = tok.apply_chat_template([{"role": "user", "content": user_msg}],
                                    add_generation_prompt=True, tokenize=True,
                                    return_dict=False)
        c = tok(completion, add_special_tokens=False)["input_ids"] + [EOS_ID]
        prompt_ids.append(p); full_ids.append(p + c)
    T_max = max(len(x) for x in full_ids)
    ids = torch.full((len(pairs), T_max), pad_id)
    for i, x in enumerate(full_ids):
        ids[i, :len(x)] = torch.tensor(x)
    plens = [len(p) for p in prompt_ids]
    m = completion_mask_from_prompt_lens(ids, plens, pad_token_id=pad_id)
    return ids, m, plens, prompt_ids, full_ids

input_ids, mask, prompt_lens, prompt_ids, full_ids = build_batch(PAD_ID)
print(f"torch {torch.__version__} | RUN_TRAINING = {RUN_TRAINING}")
print(f"batch {tuple(input_ids.shape)}, prompt lens {prompt_lens}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:00<04:14,  1.14it/s]

Loading weights:   8%|▊         | 24/290 [00:00<00:08, 33.12it/s]

Loading weights:  13%|█▎        | 39/290 [00:01<00:04, 51.76it/s]

...

Loading weights:  53%|█████▎    | 144/272 [00:00<00:00, 371.34it/s]

Loading weights:  67%|██████▋   | 182/272 [00:00<00:00, 315.21it/s]

Loading weights:  79%|███████▉  | 216/272 [00:00<00:00, 319.79it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 362.74it/s]

torch 2.13.0+cpu | RUN_TRAINING = False
batch (4, 53), prompt lens [40, 39, 36, 42]


## Exercise 1: the §3 assertion against a training-scale pair

**The exercise.** On the training box, swap `Qwen/Qwen3-8B` and `Qwen/Qwen3-1.7B` into the
§3 cross-check (kd_core's external shift-and-mask loss versus Hugging Face's internal
`model(labels=...).loss`), in bf16 with `device_map="auto"`, and confirm it still passes.
Note what you had to change (the comparison tolerance, because bf16 arithmetic is noisier
than fp32) and what you did not (any line of alignment code).

**The approach.** The point of the exercise is that the cross-check is *scale-invariant*: it
verifies a coordinate convention, not model quality, so the only thing allowed to change
with a bigger pair is the float noise floor. The plan: first execute the check live on the
lab's SmolLM2 pair, which proves the exact code path this solution ships; then the
training-scale version behind `RUN_TRAINING`, identical except for three declared
differences: the model names, the dtype (bf16, since 8B in fp32 would need about 32 GB for
weights alone, 8 billion parameters times 4 bytes), and the tolerance. The tolerance
reasoning, so the number is not magic: bf16 keeps roughly 2 to 3 significant digits per
stored value (Lab 00 §7), losses sit in the low units of nats, and the two code paths order
their reductions differently, so disagreement around `1e-3` is expected float noise and
disagreement at `1e-1` would be a real alignment bug. `3e-3` splits those cleanly.

In [2]:
def hf_vs_kdcore_loss(model_s, model_t, ids, m, atol):
    labels = ids.clone()
    labels[~m] = -100
    with torch.no_grad():
        out_s = model_s(ids, labels=labels)
        s_logits = out_s.logits
        t_logits = model_t(ids).logits
    hf_loss = float(out_s.loss)
    s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, m)
    lp = F.log_softmax(s_sh.float(), dim=-1)
    nll = -lp.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    our_loss = float(masked_mean(nll, m_sh))
    print(f"  HF model(labels=...).loss : {hf_loss:.6f}")
    print(f"  shift + mask + masked_mean: {our_loss:.6f}   |diff| = {abs(hf_loss - our_loss):.2e}")
    assert abs(hf_loss - our_loss) < atol, "kd_core alignment must reproduce HF's internal shift"
    return s_logits, t_logits

print("live demonstration on the lab's pair (fp32, tolerance 1e-4):")
s_logits, t_logits = hf_vs_kdcore_loss(student, teacher, input_ids, mask, atol=1e-4)
print("passed: identical alignment code, small pair\n")

if RUN_TRAINING:
    # The training-scale version. Three changes, all declared: names, dtype, tolerance.
    Q_TEACHER, Q_STUDENT = "Qwen/Qwen3-8B", "Qwen/Qwen3-1.7B"
    q_tok = AutoTokenizer.from_pretrained(Q_TEACHER)
    q_teacher = AutoModelForCausalLM.from_pretrained(
        Q_TEACHER, dtype=torch.bfloat16, device_map="auto").eval()
    q_student = AutoModelForCausalLM.from_pretrained(
        Q_STUDENT, dtype=torch.bfloat16, device_map="auto").eval()
    q_pids, q_fids = [], []
    for user_msg, completion in pairs:
        p = q_tok.apply_chat_template([{"role": "user", "content": user_msg}],
                                      add_generation_prompt=True, tokenize=True,
                                      return_dict=False)
        c = q_tok(completion, add_special_tokens=False)["input_ids"] + [q_tok.eos_token_id]
        q_pids.append(p); q_fids.append(p + c)
    q_pad = q_tok.pad_token_id
    T_max = max(len(x) for x in q_fids)
    q_ids = torch.full((len(pairs), T_max), q_pad)
    for i, x in enumerate(q_fids):
        q_ids[i, :len(x)] = torch.tensor(x)
    q_mask = completion_mask_from_prompt_lens(q_ids, [len(p) for p in q_pids],
                                              pad_token_id=q_pad)
    q_ids, q_mask = q_ids.to(q_student.device), q_mask.to(q_student.device)
    print("training-scale pair (bf16, tolerance 3e-3):")
    hf_vs_kdcore_loss(q_student, q_teacher, q_ids, q_mask, atol=3e-3)
    print("passed: identical alignment code, training-scale pair")
else:
    print("training-scale run gated: RUN_TRAINING = False (expected results in the next cell)")

live demonstration on the lab's pair (fp32, tolerance 1e-4):


  HF model(labels=...).loss : 2.275541
  shift + mask + masked_mean: 2.275541   |diff| = 0.00e+00
passed: identical alignment code, small pair

training-scale run gated: RUN_TRAINING = False (expected results in the next cell)


**Interpretation.** The live half executed here: on the SmolLM2 pair the two loss paths
printed above agree to well inside the fp32 tolerance of `1e-4`, which re-grounds every
shifted, masked divergence in this solutions notebook on the ecosystem's convention.

The training-scale half did not execute in this build (`RUN_TRAINING = False`); that is the
one gated item in this notebook. Expected results when you flip the flag on a machine with
the memory for an 8B teacher: both printed losses land in roughly the 1 to 4 nats range
(instruct models scoring short plausible completions), and their absolute difference lands
around `1e-4` to `1e-3`, passing the `3e-3` tolerance. What confirms the exercise's premise
is *which lines changed*: model names, dtype, pad-token lookup (Qwen3 declares a proper pad
token, so no manual `<|endoftext|>` choice is needed), and the tolerance; not one line of
`shift_for_next_token`, mask construction, or `masked_mean`. Failure signatures if you see
them: a difference of order `1e-1` or more nats means an alignment bug (most likely the mask
built over a differently rendered chat template), not float noise; a difference that grows
with sequence length points at padding entering the loss on one path; and a clean pass with
suspiciously *identical* floats suggests both paths silently ran in fp32, worth checking
because it means your training-loop memory estimate is 2x off.

## Exercise 2: find your k on your corpus

**The exercise.** Replace the four toy pairs with 200 prompts from the corpus you actually
intend to distill on, rerun the §5 truncation-bias measurement, and commit to a k with a
written justification: mass covered, tail error, storage on disk, prefill minutes.

**The approach.** The corpus this course actually distills on is the smoltalk-derived set in
`../data/lab03/` (Lab 03 trains on exactly these tensors), so that is the right corpus, and
the honest scale-down is to sample it: 16 conversations truncated to 192 tokens, roughly
1,200 supervised positions of which an evenly spaced 768 are kept for the dense
measurement (the subsample bounds memory, not information: it spans every conversation),
instead of 200 full prompts, because a 360M-parameter teacher
in fp32 on CPU prefills at tens of tokens per second and the build budget is five minutes
per cell. A sample this size estimates *mean* retained mass and mean KL bias tightly enough
to choose between k=16, 64, and 128; what it cannot see is the extreme tail of hard
positions, which is why the justification below quotes the measurement as a pre-flight
estimate, the same role Unit 04 gives it. The plan: run teacher and student on the sample,
shift into prediction coordinates, and let `topk_truncation_bias` produce the whole
decision table (dense KL, both estimators, retained mass, relative errors); then price
storage with `bytes_per_token_cache` and prefill time at the lab's measured 2053 tokens per
second rate for a 20B-class teacher, since the point of choosing k is the bill it implies at
training scale.

In [3]:
d = torch.load("../data/lab03/eval.pt")
keep = [i for i, pl in enumerate(d["prompt_lens"]) if pl < 96][:16]
ids_c = d["input_ids"][keep][:, :192]
mask_c = d["mask"][keep][:, :192]

# Memory discipline: a [16, 192, 49152] fp32 logit tensor is ~0.6 GB, and the bias
# measurement needs several dense intermediates of the same size. So process 2 rows at a
# time, keep only the supervised positions, and subsample those evenly to 768, which caps
# the densest intermediate near 150 MB without changing what is being estimated.
import gc
s_rows, t_rows = [], []
for i in range(0, len(ids_c), 2):
    with torch.no_grad():
        tl_c = teacher(ids_c[i:i+2]).logits
        sl_c = student(ids_c[i:i+2]).logits
    s_sh_c, t_sh_c, m_sh_c = shift_for_next_token(sl_c, tl_c, mask_c[i:i+2])
    s_rows.append(s_sh_c[m_sh_c].clone()); t_rows.append(t_sh_c[m_sh_c].clone())
    del tl_c, sl_c, s_sh_c, t_sh_c
s_all, t_all = torch.cat(s_rows), torch.cat(t_rows)
n_total = s_all.shape[0]
sub = torch.linspace(0, n_total - 1, steps=min(768, n_total)).long()
s_sh, t_sh = s_all[sub][None], t_all[sub][None]     # [1, N_kept, V]
m_sh = torch.ones(1, s_sh.shape[1], dtype=torch.bool)
del s_rows, t_rows, s_all, t_all
gc.collect()
print(f"corpus sample: {tuple(ids_c.shape)}, supervised positions {n_total} "
      f"(subsampled evenly to {int(m_sh.sum())})")

rows = topk_truncation_bias(s_sh, t_sh, m_sh, ks=(1, 4, 16, 64, 128))
dense = rows[0]["dense_kl"]
print(f"\n{'k':>4} {'mass kept':>10} {'renorm KL':>10} {'tail KL':>9} {'renorm err':>11} "
      f"{'tail err':>9}   dense KL = {dense:.4f}")
for r in rows:
    print(f"{r['k']:>4} {r['mean_mass_covered']:>10.4f} {r['renorm_kl']:>10.4f} "
          f"{r['tail_bucket_kl']:>9.4f} {r['renorm_rel_err']:>10.1%} {r['tail_rel_err']:>8.1%}")

print("\ncost of the candidate ks, per 1M cached tokens (V=49,152, this pair's vocabulary):")
PREFILL_TOKS_PER_S = 2053   # the lab's measured prefill rate for a 20B-class teacher
for k in (16, 64, 128):
    s = bytes_per_token_cache(49152, k=k)
    print(f"  k={k:>4}: {s['topk_gb_per_1M_tokens']:.3f} GB on disk "
          f"({s['compression']:.0f}x vs dense), "
          f"prefill ~{1e6 / PREFILL_TOKS_PER_S / 60:.1f} min at {PREFILL_TOKS_PER_S} tok/s")

k64 = next(r for r in rows if r["k"] == 64)
assert k64["mean_mass_covered"] > 0.99, "k=64 keeps over 99% of teacher mass on this corpus"
assert abs(k64["renorm_rel_err"]) < 0.15 and abs(k64["tail_rel_err"]) < 0.15, \
    "both estimators sit within 15% of the dense KL at k=64"
assert all(r["renorm_kl"] >= dense - 1e-6 for r in rows), "renorm overstates on this corpus"
assert all(r["tail_bucket_kl"] <= dense + 1e-6 for r in rows), "tail bucket understates"
print("\nchecked: the decision table exists for this corpus, and k=64 clears both gates")

corpus sample: (16, 192), supervised positions 1246 (subsampled evenly to 768)



   k  mass kept  renorm KL   tail KL  renorm err  tail err   dense KL = 0.2835
   1     0.7721     0.6340    0.1321     123.6%   -53.4%
   4     0.9340     0.3705    0.2170      30.7%   -23.5%
  16     0.9801     0.3094    0.2583       9.1%    -8.9%
  64     0.9936     0.2917    0.2743       2.9%    -3.3%
 128     0.9964     0.2880    0.2781       1.6%    -1.9%

cost of the candidate ks, per 1M cached tokens (V=49,152, this pair's vocabulary):
  k=  16: 0.098 GB on disk (1003x vs dense), prefill ~8.1 min at 2053 tok/s
  k=  64: 0.386 GB on disk (255x vs dense), prefill ~8.1 min at 2053 tok/s
  k= 128: 0.770 GB on disk (128x vs dense), prefill ~8.1 min at 2053 tok/s

checked: the decision table exists for this corpus, and k=64 clears both gates


**Interpretation, and the written justification the exercise asks for.** The committed
choice: **k = 64, with the tail bucket kept**. The case, each number from the table above:
(1) *mass covered*: at k=64 the retained mass clears 0.99 (asserted), so at most one percent
of teacher probability is being summarised rather than stored; (2) *bias*: both estimators
sit within the asserted 15% of the dense KL at k=64, bracketing the truth from opposite
sides exactly as the lab's §5 found (renormalising overstates, the tail bucket understates,
both asserted across every k), and keeping the tail bucket costs one float per position while
removing the estimator's dependence on pretending the tail is empty; (3) *disk*: the printed
cost line prices k=64 at well under half a gigabyte per million tokens, roughly 250x smaller
than dense, so a 100M-token corpus caches in tens of gigabytes, which fits the storage a
single training box actually has; (4) *prefill*: about 8 minutes per million tokens at the
lab's measured rate, so the cache for that 100M-token corpus is a half-day batch job you pay
once, not a per-epoch cost. Why not the neighbours: k=16 leaves visibly more mass on the
table and multiplies the estimator errors (read its row), for a storage saving that is not
the bottleneck; k=128 halves the residual error but doubles the cache for a bias that is
already inside the noise of a training run. One scope caveat, stated rather than hidden:
this table is 16 conversations from one corpus for one pair; the whole point of the exercise
is that the table, not the conclusion, is the reusable artifact, and a different corpus or a
flatter teacher re-runs the same twenty lines and may land on a different k.

## Exercise 3: break the padding on purpose

**The exercise.** Re-pad the §2 batch with `pad_token_id = eos_token_id` and watch which
assertion fails. Explain the downstream symptom in a trained student (§2 named it: a student
never supervised on EOS never learns to stop).

**The approach.** The trap being demonstrated: `completion_mask_from_prompt_lens` excludes
padding by *token id*, so if padding and EOS share an id, the exclusion cannot tell them
apart and strips the one real EOS at the end of every completion out of supervision. The lab
guards against this up front (`assert PAD_ID != EOS_ID`); this solution deliberately skips
the guard, rebuilds the batch padded with EOS, and re-runs the lab's own §2 audit
assertions inside a `try/except AssertionError` so the notebook can display which one fires
without crashing. Then it pinpoints the damage: for each row, exactly which supervised
positions were lost relative to the correctly padded batch.

In [4]:
bad_ids, bad_mask, bad_plens, bad_pids, bad_fids = build_batch(pad_id=EOS_ID)

failures = []
for i, (p, f) in enumerate(zip(bad_pids, bad_fids)):
    n_comp = len(f) - len(p)
    try:
        assert int(bad_mask[i].sum()) == n_comp, f"row {i}: mask must cover exactly the completion"
        assert not bad_mask[i, :len(p)].any(), f"row {i}: no prompt token may be supervised"
        assert bad_mask[i, len(f) - 1], f"row {i}: the EOS token must be supervised"
    except AssertionError as e:
        failures.append(str(e))
        print(f"AUDIT FAILURE -> {e}")

# Pinpoint the damage: which supervised positions vanished, row by row.
print(f"\n{'row':>4} {'good mask sum':>14} {'bad mask sum':>13}   lost position(s)")
for i, f in enumerate(full_ids):
    lost = torch.nonzero(mask[i] & ~bad_mask[i]).squeeze(-1).tolist()
    lost_toks = [repr(tok.decode([int(input_ids[i, j])])) for j in lost]
    print(f"{i:>4} {int(mask[i].sum()):>14} {int(bad_mask[i].sum()):>13}   "
          f"{list(zip(lost, lost_toks))}")

assert len(failures) == len(pairs), "every row's audit fails when padding shares the EOS id"
assert all("cover exactly the completion" in msg for msg in failures), \
    "the count audit fires first: each mask is short by exactly its EOS"
assert all(not bad_mask[i, len(f) - 1] for i, f in enumerate(bad_fids)), \
    "and the missing position is precisely the final EOS of every row"
lost_all = [torch.nonzero(mask[i] & ~bad_mask[i]).squeeze(-1).tolist() for i in range(len(pairs))]
assert all(lost == [len(f) - 1] for lost, f in zip(lost_all, full_ids)), \
    "no other position was touched: the damage is exactly one EOS per sequence"
print("\nchecked: pad = EOS silently unsupervises exactly the EOS of every sequence")

AUDIT FAILURE -> row 0: mask must cover exactly the completion
AUDIT FAILURE -> row 1: mask must cover exactly the completion
AUDIT FAILURE -> row 2: mask must cover exactly the completion
AUDIT FAILURE -> row 3: mask must cover exactly the completion

 row  good mask sum  bad mask sum   lost position(s)
   0             13            12   [(52, "'<|im_end|>'")]
   1              4             3   [(42, "'<|im_end|>'")]
   2             12            11   [(47, "'<|im_end|>'")]
   3             10             9   [(51, "'<|im_end|>'")]

checked: pad = EOS silently unsupervises exactly the EOS of every sequence


**Interpretation.** The audit output shows the first §2 assertion to fire is the count
check, "mask must cover exactly the completion", on every row (asserted): each row's mask
comes up exactly one position short. The pinpoint table shows the missing position is always
the last token of the sequence and decodes to `<|im_end|>`, the real EOS, and the final
assertion confirms *nothing else changed*: prompts still excluded, completion bodies still
supervised, shapes identical. That is what makes this bug dangerous rather than loud. The
loss still computes, still goes down, and every summary statistic moves by roughly one part
in ten (one lost position out of a dozen supervised ones per row here, far less on longer
completions).

The downstream symptom, mechanically: the student receives gradient on every completion
token except the one that says "stop". Trained that way, it learns the distribution of
answer *content* but never raises the probability of emitting EOS after the answer is
complete, so at inference it finishes the answer and keeps sampling, running to the length
limit, often restarting a new turn of conversation with itself. The failure is invisible in
training curves and obvious in the product. It is the same failure
`kd_core.onpolicy_mask`'s docstring warns about from the generation side (supervising
*past* EOS teaches padding; here, failing to supervise *at* EOS teaches never stopping),
and the two-line defence is the lab's: pick a pad token that cannot appear in a templated
conversation, and keep the `PAD_ID != EOS_ID` assert where it can fire before any training
job spends a dollar.

## Exercise 4: fertility as a cost model

**The exercise.** Tokenize 1,000 lines of your corpus with all three §1 tokenizers and
compute total tokens. A teacher whose tokenizer is 15% more fertile (fertility is tokens
produced per byte of text) pays 15% more prefill compute, 15% more cache storage, and 15%
more per-token decode on every rollout it ever scores. Which of the §1 tokenizers would you
want owning your corpus?

**The approach.** Use the actual course corpus again: decode 1,000 conversations from
`../data/lab03/train.pt` back to plain text (dropping the template's special tokens, so all
three tokenizers see the same natural-language payload rather than markers only one of them
knows), then encode the same 1,000 texts with GPT-2, Qwen2.5, and SmolLM2 tokenizers and
compare total token counts. The winner is the tokenizer with the fewest tokens for the same
bytes, and the margins convert directly into cost multipliers on all three meters at once,
because every meter in a distillation pipeline (prefill FLOPs, cache rows, decode steps) is
denominated in tokens.

In [5]:
tr = torch.load("../data/lab03/train.pt")
texts = [tok.decode([t for t in row.tolist() if t != PAD_ID], skip_special_tokens=True)
         for row in tr["input_ids"][:1000]]
total_bytes = sum(len(t.encode()) for t in texts)
print(f"corpus sample: {len(texts)} conversations, {total_bytes:,} bytes of text")

tokenizers = {"gpt2": gpt2_tok, "Qwen2.5": qwen_tok, "SmolLM2": tok}
totals = {}
for name, tk in tokenizers.items():
    totals[name] = sum(len(tk(t, add_special_tokens=False)["input_ids"]) for t in texts)

best = min(totals, key=totals.get)
print(f"\n{'tokenizer':>10} {'total tokens':>13} {'tokens/byte':>12} {'cost vs best':>13}")
for name, n in sorted(totals.items(), key=lambda kv: kv[1]):
    print(f"{name:>10} {n:>13,} {n / total_bytes:>12.4f} {n / totals[best] - 1:>+12.1%}")

spread = max(totals.values()) / min(totals.values())
assert len(set(totals.values())) == 3, "three tokenizers, three different bills"
assert spread > 1.02, "the fertility gap is real money, not rounding"
print(f"\nchecked: same {total_bytes:,} bytes, three bills; "
      f"the most fertile tokenizer charges {spread - 1:+.1%} over {best}")

corpus sample: 1000 conversations, 1,044,230 bytes of text



 tokenizer  total tokens  tokens/byte  cost vs best
   Qwen2.5       223,014       0.2136        +0.0%
   SmolLM2       238,525       0.2284        +7.0%
      gpt2       247,630       0.2371       +11.0%

checked: same 1,044,230 bytes, three bills; the most fertile tokenizer charges +11.0% over Qwen2.5


**Interpretation.** The printed table is a price list for the identical text. On this
corpus the cheapest tokenizer wins by the printed margin over the most expensive (the
assertion only demands the gap exceed 2%, because the exact spread is a property of this
corpus; the printed value is the real number for this one). Read the "cost vs best" column
as a multiplier that applies three times to the same run: a tokenizer that is x% more
fertile on your corpus makes the teacher prefill x% more tokens when building the cache,
makes the cache itself x% larger at any fixed k (the cache stores one row per token), and,
worst because it recurs, makes every on-policy rollout x% longer to score, every epoch,
for the life of the project. Which would I want owning this corpus? The winner by the
table, with one caveat that matters more than the percentage: the tokenizer that "owns" the
corpus in a distillation is fixed by which teacher and student you can actually use, so in
practice this measurement runs in the other direction. You do not pick a tokenizer for its
fertility; you *price* the pair you are stuck with, and a candidate teacher whose tokenizer
is 15% more fertile on your corpus needs to be more than 15% better per token to be worth
it. That is why fertility-on-your-corpus belongs in the same pre-flight checklist as
Exercise 2's retained-mass table: both are one-screen measurements that turn a model-card
decision into an arithmetic one.

## Exercise 5: sorted logits survive re-tokenization

**The exercise.** For a boundary position shared by both §6 tokenizers, take each model's
next-token distribution, sort each probability vector in descending order, and compare the
sorted tails. Sorting throws away which token each probability belongs to, and that is
exactly what makes the comparison legal across two different vocabularies. This is the
first step of ULD.

**The approach.** The §6 measurement found which byte positions of a string are token
boundaries under *both* GPT-2 and SmolLM2. At such a position both models have read exactly
the same text, so "what comes next" is the same question for both, even though their answer
spaces (vocabularies of 50,257 and 49,152 entries naming different strings) are
incompatible. The plan: pick a shared boundary mid-string, run GPT-2 (the small 124M base
model) and SmolLM2-135M-Instruct on their own tokenizations of the identical prefix, and
demonstrate two things in sequence. First, the *negative* result stated by §6: a token-space
KL between the two next-token distributions is not merely inaccurate but undefined, which in
tensor terms surfaces as a hard shape error, caught and displayed. Second, the *positive*
result: sorting each probability vector in descending order maps both models into a shared
coordinate system ("probability of the rank-r choice"), where head-to-head comparison is
legal, bounded, and interpretable as a statement about confidence profiles.

In [6]:
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float32).eval()

sample = ("Distillation transfers the teacher's distribution, not its weights. "
          "273 GB/s is the budget; design the pipeline around prefill.")

def boundary_set(tk):
    enc = tk(sample, add_special_tokens=False, return_offsets_mapping=True)
    return set(e for _, e in enc["offset_mapping"])

shared = sorted(boundary_set(gpt2_tok) & boundary_set(tok))
b = [s for s in shared if 40 < s < 70][0]         # a shared boundary mid-sentence
ctx = sample[:b]
print(f"shared boundary at byte {b}; both models read: ...{ctx[-32:]!r}")

with torch.no_grad():
    p_g = F.softmax(gpt2_model(torch.tensor([gpt2_tok(ctx)["input_ids"]])).logits[0, -1], -1)
    p_s = F.softmax(student(torch.tensor([tok(ctx)["input_ids"]])).logits[0, -1], -1)
print(f"gpt2 vocab {p_g.shape[0]:,} | SmolLM2 vocab {p_s.shape[0]:,}")

# Negative result first: token-space KL across vocabularies is undefined, not just noisy.
kl_failed = False
try:
    _ = (p_g * (p_g.log() - p_s.log())).sum()
except RuntimeError as e:
    kl_failed = True
    print(f"\ntoken-space KL raises: {str(e)[:72]}...")
assert kl_failed, "distributions over different outcome spaces cannot meet in a KL"

# Positive result: sort away the token identities and compare rank profiles.
srt_g = p_g.sort(descending=True).values
srt_s = p_s.sort(descending=True).values
L = min(len(srt_g), len(srt_s))
l1_sorted = float((srt_g[:L] - srt_s[:L]).abs().sum())

print(f"\n{'rank':>5} {'gpt2 sorted p':>14} {'SmolLM2 sorted p':>17}")
for rnk in (0, 1, 2, 4, 9, 63):
    print(f"{rnk + 1:>5} {float(srt_g[rnk]):>14.5f} {float(srt_s[rnk]):>17.5f}")
print(f"tail mass beyond rank 64: gpt2 {float(srt_g[64:].sum()):.4f}, "
      f"SmolLM2 {float(srt_s[64:].sum()):.4f}")
print(f"L1 distance between sorted vectors: {l1_sorted:.4f}  (bounded by 2)")

assert (srt_g[:-1] >= srt_g[1:]).all() and (srt_s[:-1] >= srt_s[1:]).all()
assert 0.0 <= l1_sorted <= 2.0, "sorted-vector L1 is a bounded, well-defined number"
assert float(srt_g[:L].sum()) > 0.999 and float(srt_s[:L].sum()) > 0.999, \
    "truncating to the shorter vocabulary discards negligible sorted mass"
print("\nchecked: token-space comparison is undefined; rank-space comparison is a number")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1863.06it/s]

shared boundary at byte 49; both models read: ..."sfers the teacher's distribution"


gpt2 vocab 50,257 | SmolLM2 vocab 49,152

token-space KL raises: The size of tensor a (50257) must match the size of tensor b (49152) at ...

 rank  gpt2 sorted p  SmolLM2 sorted p
    1        0.52928           0.35324
    2        0.13110           0.16439
    3        0.03535           0.11063
    5        0.01717           0.03367
   10        0.00998           0.01520
   64        0.00054           0.00052
tail mass beyond rank 64: gpt2 0.0912, SmolLM2 0.0693
L1 distance between sorted vectors: 0.4368  (bounded by 2)

checked: token-space comparison is undefined; rank-space comparison is a number


**Interpretation.** The negative result printed first is §6's claim made concrete: the
KL between the two next-token distributions fails with a tensor-shape error, because 50,257
outcomes cannot be paired with 49,152 outcomes, and no epsilon or projection makes that
subtraction meaningful; renaming the error away (by padding one vector, say) would compare
"probability of GPT-2's token 17" with "probability of SmolLM2's token 17", two unrelated
strings.

The positive result is the rank table. After sorting, position r means the same thing for
both models ("the probability of the r-th most likely continuation"), so the columns can be
read against each other: both models, having read the identical prefix, concentrate most
mass in their top few ranks and hold under a tenth of it beyond rank 64, and the
truncation assertion shows cutting both vectors to the shorter vocabulary's length discards
under a tenth of a percent of mass, so the vocabulary-size mismatch stops mattering in rank
space. The L1 distance between the sorted vectors, a number that cannot exceed 2, lands
well under 0.5 here: the two models disagree noticeably about *how confident* to be (read
the rank-1 row), while both agree the position is a moderately peaked one. That is exactly
what sorting preserves: the confidence profile, how mass decays with rank, with every token
identity deliberately forgotten. It is also exactly what sorting costs: this distance can
never see that the models might favour *different* continuations with identical confidence.
ULD, whose first step this is, accepts that trade to get a well-defined training signal
across vocabularies, and Unit 10 pairs it with alignment machinery that recovers some of
the discarded identity information. The exercise's point stands verified: the only
quantities that survive re-tokenization are the ones that never mention token ids, and you
have now computed one.